In [25]:
import requests
import pandas as pd
import time

BASE_URL = "https://product.kt.com/wDic/getOptionItemListAjax.ajax"
HEADERS = {"User-Agent": "Mozilla/5.0"}


def fetch_page(cate_code, filter_code, option_code, page_no, list_size=10):
    params = {
        "cate_code": cate_code,
        "pageNo": page_no,
        "listSize": list_size,
        "filter_code": filter_code,
        "option_code": option_code,
    }
    resp = requests.get(BASE_URL, params=params, headers=HEADERS)
    resp.raise_for_status()
    return resp.text


def crawl_all_pages(cate_code, filter_code, option_code, list_size=10, max_pages=50, sleep_sec=0.3):
    all_dfs = []
    page_no = 1

    while page_no <= max_pages:
        html = fetch_page(cate_code, filter_code, option_code, page_no, list_size)
        df = parse_kt_plan_list(html)

        if df.empty:
            break

        df["page"] = page_no
        all_dfs.append(df)
        page_no += 1
        time.sleep(sleep_sec)

    if not all_dfs:
        return pd.DataFrame()
    return pd.concat(all_dfs, ignore_index=True)


df = crawl_all_pages(cate_code="6002", filter_code="191", option_code="260")
df

C:\Users\NT551_11TH\AppData\Local\Temp\ipykernel_23476\3329661429.py:40: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(all_dfs, ignore_index=True)


,carrier,plan_name,monthly_fee,monthly_fee_discounted,item_code,page
0,KT,초이스 더블,120000.0,81600.0,1691,1
1,KT,초이스,90000.0,61200.0,1681,1
2,KT,베이직,28900.0,19651.0,1693,1
3,KT,베이직(이월),37000.0,25160.0,1694,1
4,KT,음성,12100.0,8228.0,1697,1
5,KT,요고 모바일,30000.0,NaN,1567,1
6,KT,키즈,13000.0,8840.0,1695,1
7,KT,웰컴,39000.0,26520.0,1577,1
8,KT,스마트기기,16500.0,NaN,1696,1
9,KT,데이터투게더,11000.0,NaN,1698,1


In [ ]:
import time

BASE_URL = "https://product.kt.com"
HEADERS = {"User-Agent": "Mozilla/5.0"}


def resolve_detail_url(item_code, cate_code, filter_code, option_code, page_size=10):
    if item_code is None:
        return None

    item_url = (
        f"{BASE_URL}/wDic/productDetail.do"
        f"?ItemCode={item_code}&CateCode={cate_code}"
        f"&filter_code={filter_code}&option_code={option_code}&pageSize={page_size}"
    )
    html_text = requests.get(item_url, headers=HEADERS).text

    match = re.search(r'(htmlUploadType_\d+\.html)', html_text)
    if not match:
        return None

    return f"{BASE_URL}/static/prodetail/{item_code}/web/{match.group(1)}"


def add_detail_urls(df, cate_code, filter_code, option_code, sleep_sec=0.3):
    urls = []
    for code in df["item_code"]:
        urls.append(resolve_detail_url(code, cate_code, filter_code, option_code))
        time.sleep(sleep_sec)
    df["detail_url"] = urls
    return df


df = add_detail_urls(df, cate_code="6002", filter_code="191", option_code="260")
df

,carrier,plan_name,monthly_fee,monthly_fee_discounted,item_code,page,detail_url
0,KT,초이스 더블,120000.0,81600.0,1691,1,https://product.kt.com/static/prodetail/1691/w...
1,KT,초이스,90000.0,61200.0,1681,1,None
2,KT,베이직,28900.0,19651.0,1693,1,None
3,KT,베이직(이월),37000.0,25160.0,1694,1,https://product.kt.com/static/prodetail/1694/w...
4,KT,음성,12100.0,8228.0,1697,1,https://product.kt.com/static/prodetail/1697/w...
5,KT,요고 모바일,30000.0,NaN,1567,1,None
6,KT,키즈,13000.0,8840.0,1695,1,None
7,KT,웰컴,39000.0,26520.0,1577,1,https://product.kt.com/static/prodetail/1577/w...
8,KT,스마트기기,16500.0,NaN,1696,1,None
9,KT,데이터투게더,11000.0,NaN,1698,1,https://product.kt.com/static/prodetail/1698/w...


In [31]:
df[[ "plan_name", "detail_url"]]

,plan_name,detail_url
0,초이스 더블,https://product.kt.com/static/prodetail/1691/w...
1,초이스,None
2,베이직,None
3,베이직(이월),https://product.kt.com/static/prodetail/1694/w...
4,음성,https://product.kt.com/static/prodetail/1697/w...
5,요고 모바일,None
6,키즈,None
7,웰컴,https://product.kt.com/static/prodetail/1577/w...
8,스마트기기,None
9,데이터투게더,https://product.kt.com/static/prodetail/1698/w...


In [53]:
detail_soup = BeautifulSoup(detail_html, 'html.parser')

In [54]:
detail_soup

<div class="inner">
<div class="location">
<script type="text/javascript">
    if(window.location.pathname.includes("productDetail.do")){
        kt.locationBar();
    }
</script>
</div>
<div class="N-cHead-section">
<div class="N-cHead-column">
<div class="N-cHeadline-cover">
<h1 class="N-cHeadline">초이스<br/>
더블</h1>
<div class="N-head-btn-section">
<div class="N-head-btn-column sns"><button class="opener" onclick="KT_product_trackClicks('KT-개인_상품_모바일_모바일_요금제','^KT-개인_상품_모바일_모바일_요금제_초이스_더블^기본정보^공유하기');" type="button"><img alt="공유하기" src="/static/prodetail/N_version/web/images/common/icon_N_sns.png"/></button>
<div class="N-head-btn-area"><a class="icon kakao" href="javascript:void(0);" onclick="sns.gokakao('1691','폰케어 초이스'); KT_product_trackClicks('KT-개인_상품_모바일_모바일_요금제','^KT-개인_상품_모바일_모바일_요금제_초이스_더블^기본정보^공유하기_카카오톡');" title="새창열림">카카오톡</a><a class="icon facebook" href="javascript:void(0);" onclick="sns.goFaceBook('폰케어 초이스','https://product.kt.com/wDic/productDetail.do?ItemCode=1691'); 